# Chess tokenizer exploration

First pass at a compositional SAN/movetext tokenizer for the Lumbras PGN splits.


In [ ]:
import re
from dataclasses import dataclass
from typing import Iterable

# Chess-aware SAN tokenizer for PGN movetext.
# Goal: expressive, compositional tokens without one token per possible move/position.
#
# Examples:
#   Nbd7      -> PIECE_N SRC_FILE_b TO_d7 <EOM>
#   exd8=Q+   -> PAWN SRC_FILE_e CAPTURE TO_d8 PROMO_Q CHECK <EOM>
#   O-O       -> CASTLE_KINGSIDE <EOM>
#   R1e2      -> PIECE_R SRC_RANK_1 TO_e2 <EOM>
#   Qh5#      -> PIECE_Q TO_h5 MATE <EOM>

FILES = "abcdefgh"
RANKS = "12345678"
PIECES = set("KQRBN")
RESULTS = {"1-0": "RESULT_WHITE", "0-1": "RESULT_BLACK", "1/2-1/2": "RESULT_DRAW", "*": "RESULT_UNKNOWN"}

# Strip PGN comments, recursive annotation variations, and common numeric annotation glyphs.
COMMENT_RE = re.compile(r"\{[^}]*\}|;[^\n]*")
NAG_RE = re.compile(r"\$\d+")
MOVE_NUMBER_RE = re.compile(r"^\d+\.(?:\.\.)?$")

SAN_RE = re.compile(
    r"^"
    r"(?P<piece>[KQRBN])?"          # absent means pawn
    r"(?P<src_file>[a-h])?"         # disambiguation or pawn capture source file
    r"(?P<src_rank>[1-8])?"         # disambiguation
    r"(?P<capture>x)?"
    r"(?P<target>[a-h][1-8])"
    r"(?:=(?P<promo>[QRBN]))?"
    r"(?P<suffix>[+#])?"
    r"(?P<annotation>[!?]+)?"
    r"$"
)


def strip_variations(text: str) -> str:
    """Remove parenthesized PGN variations, including nested ones."""
    out = []
    depth = 0
    for ch in text:
        if ch == "(":
            depth += 1
            continue
        if ch == ")" and depth:
            depth -= 1
            continue
        if depth == 0:
            out.append(ch)
    return "".join(out)


def clean_movetext(text: str) -> str:
    """Remove metadata-adjacent PGN clutter while preserving SAN move text."""
    text = COMMENT_RE.sub(" ", text)
    text = strip_variations(text)
    text = NAG_RE.sub(" ", text)
    return text.replace("\n", " ")


def tokenize_san(san: str, add_eom: bool = True) -> list[str]:
    """Tokenize one SAN move into compositional chess tokens.

    <EOM> marks the boundary after a complete move, which makes detokenization
    and legality-check handoff to an external chess environment unambiguous.
    Result tokens do not receive <EOM> because they terminate the game, not a move.
    """
    san = san.strip()
    if not san:
        return []

    # Normalize zeros sometimes used for castling.
    san = san.replace("0", "O") if san in {"0-O", "0-O-O", "0-0", "0-0-0"} else san

    if san in RESULTS:
        return [RESULTS[san]]

    # Castling can carry check/checkmate suffixes.
    if san.startswith("O-O-O"):
        suffix = san[5:]
        tokens = ["CASTLE_QUEENSIDE"]
    elif san.startswith("O-O"):
        suffix = san[3:]
        tokens = ["CASTLE_KINGSIDE"]
    else:
        # Drop common trailing annotations before structural parse.
        san_core = re.sub(r"[!?]+$", "", san)
        match = SAN_RE.match(san_core)
        if not match:
            return ["UNK_SAN", f"RAW_{san}"]

        gd = match.groupdict()
        piece = gd["piece"] or "P"
        tokens = ["PAWN" if piece == "P" else f"PIECE_{piece}"]

        if gd["src_file"]:
            # For pawn captures this is the source file; for pieces it is disambiguation.
            tokens.append(f"SRC_FILE_{gd['src_file']}")
        if gd["src_rank"]:
            tokens.append(f"SRC_RANK_{gd['src_rank']}")
        if gd["capture"]:
            tokens.append("CAPTURE")

        tokens.append(f"TO_{gd['target']}")

        if gd["promo"]:
            tokens.append(f"PROMO_{gd['promo']}")

        suffix = gd["suffix"] or ""

    if "+" in suffix:
        tokens.append("CHECK")
    if "#" in suffix:
        tokens.append("MATE")
    if add_eom:
        tokens.append("<EOM>")

    return tokens


def tokenize_movetext(movetext: str, include_turn_tokens: bool = True) -> list[str]:
    """Tokenize PGN movetext into a flat model-ready token sequence."""
    cleaned = clean_movetext(movetext)
    tokens: list[str] = []
    ply = 0

    for raw in cleaned.split():
        raw = raw.strip()
        if not raw:
            continue

        # Split forms like "1.e4" or "1...Nf6". If a token explicitly
        # uses black-to-move notation, align the local ply counter with black.
        explicit_black_move = bool(re.match(r"^\d+\.\.\.", raw))
        raw = re.sub(r"^\d+\.{1,3}", "", raw)
        if not raw or MOVE_NUMBER_RE.match(raw):
            continue
        if explicit_black_move and ply % 2 == 0:
            ply += 1

        move_tokens = tokenize_san(raw)
        if include_turn_tokens and move_tokens and not move_tokens[0].startswith("RESULT_"):
            tokens.append("WHITE" if ply % 2 == 0 else "BLACK")
            ply += 1
        tokens.extend(move_tokens)
        if move_tokens and move_tokens[0].startswith("RESULT_"):
            break

    return tokens


def pgn_to_movetext(pgn: str) -> str:
    """Drop PGN tag-pair metadata and return only movetext lines."""
    return "\n".join(line for line in pgn.splitlines() if not line.startswith("["))


# Quick sanity checks / examples.
examples = ["Nbd7", "exd8=Q+", "O-O", "O-O-O#", "R1e2", "Qh5#", "1.e4", "1...Nf6"]
for ex in examples:
    print(f"{ex:10s} -> {tokenize_movetext(ex) if ex[0].isdigit() else tokenize_san(ex)}")

sample_movetext = "1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. Bxc6 dxc6 5. O-O Nf6 1/2-1/2"
print("\nsample:", tokenize_movetext(sample_movetext))


## Network representation sketch

We will treat each chess move as a fixed-width **move packet** rather than a single opaque move token.

For now:

- `MOVE_SEQ_LEN = 8`
- each row is one full SAN move represented as token ids
- `<EOM>` marks the end of the real move
- `<PAD>` fills the remaining packet positions
- `<PAD>` is id `0`, so it is easy to mask out of losses / legality logic

Examples:

```text
Nbd7     -> [PIECE_N, SRC_FILE_b, TO_d7, <EOM>, <PAD>, <PAD>, <PAD>, <PAD>]
exd8=Q+  -> [PAWN, SRC_FILE_e, CAPTURE, TO_d8, PROMO_Q, CHECK, <EOM>, <PAD>]
O-O      -> [CASTLE_KINGSIDE, <EOM>, <PAD>, <PAD>, <PAD>, <PAD>, <PAD>, <PAD>]
```

Training table shape:

```text
moves_ids: uint16[num_moves, 8]
```

For sequence modeling, we likely form examples as previous move packets predicting the next move packet:

```text
x: int64[batch, context_moves, 8]
y: int64[batch, 8]
```

For a diffusion-style move head, `y` is the denoising target over the 8 packet slots. Each slot predicts a categorical distribution over the vocabulary. `<PAD>` slots after `<EOM>` should be included as structural targets but masked/handled distinctly from legal chess tokens.

Important: this is still only a notation representation. Legal move validation/masking will be handled by an external chess environment later.


In [ ]:
from pathlib import Path
import sys
import numpy as np

# Use the reusable tokenizer module from ../src
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from chessgm.tokenizer import ChessTokenizer, VOCAB

MOVE_SEQ_LEN = 8
tok = ChessTokenizer()

PAD_ID = tok.token_to_id["<PAD>"]
EOM_ID = tok.token_to_id["<EOM>"]

print("vocab_size:", len(VOCAB))
print("PAD_ID:", PAD_ID)
print("EOM_ID:", EOM_ID)

def encode_move_packet(san: str, seq_len: int = MOVE_SEQ_LEN) -> np.ndarray:
    """Encode one SAN move as a fixed-width packet of token ids."""
    encoded = tok.encode_move(san)
    if len(encoded.ids) > seq_len:
        raise ValueError(f"Move {san!r} needs {len(encoded.ids)} ids > seq_len={seq_len}: {encoded.tokens}")
    return np.array(encoded.ids + [PAD_ID] * (seq_len - len(encoded.ids)), dtype=np.uint16)

def decode_move_packet(packet: np.ndarray) -> str:
    """Decode one fixed-width packet back to SAN, ignoring PAD."""
    ids = [int(i) for i in packet if int(i) != PAD_ID]
    return tok.decode_move(ids)

examples = ["e4", "Nbd7", "exd8=Q+", "O-O", "O-O-O#"]
packets = np.stack([encode_move_packet(m) for m in examples])

for move, packet in zip(examples, packets):
    tokens = tok.decode_ids(packet.tolist())
    print(f"{move:8s} -> ids={packet.tolist()} tokens={tokens} decoded={decode_move_packet(packet)}")

print("\npacket table shape:", packets.shape, packets.dtype)


In [ ]:
# Mock model batch construction.
# Suppose a game has N move packets. A training example can be:
#   x = previous context window of move packets
#   y = next move packet

CONTEXT_MOVES = 4
mock_game = ["e4", "e5", "Nf3", "Nc6", "Bb5", "a6", "Bxc6", "dxc6", "O-O"]
game_packets = np.stack([encode_move_packet(m) for m in mock_game])

xs = []
ys = []
for i in range(CONTEXT_MOVES, len(game_packets)):
    xs.append(game_packets[i-CONTEXT_MOVES:i])
    ys.append(game_packets[i])

x = np.stack(xs).astype(np.int64)
y = np.stack(ys).astype(np.int64)

print("x shape:", x.shape, "# [batch, context_moves, move_seq_len]")
print("y shape:", y.shape, "# [batch, move_seq_len]")
print("first target ids:", y[0].tolist())
print("first target tokens:", tok.decode_ids(y[0].tolist()))
print("first target SAN:", decode_move_packet(y[0]))

# Useful masks:
pad_mask = (y == PAD_ID)          # slots that are padding
nonpad_mask = (y != PAD_ID)       # slots that participate in move structure
post_eom_is_pad = y[:, -1] == PAD_ID

print("pad_mask shape:", pad_mask.shape)
print("nonpad slots in first target:", nonpad_mask[0].tolist())


## Baseline network: pure autoregressive packet transformer

First network sketch, intentionally simple:

- no thinker yet
- no board-state embedding yet
- input is only previous move packets
- output is the next move packet `[8]`
- main architectural constraint: RoPE position increments by `move_index`, not by token slot

Input / output:

```text
x_ids: [batch, context_moves, move_seq_len]
y_ids: [batch, move_seq_len]
logits: [batch, move_seq_len, vocab_size]
```

RoPE rule:

```text
position(move_i, slot_j) = i
```

So all 8 token slots inside one move packet receive the same rotary position.


In [ ]:
# Minimal PyTorch sketch. This cell requires torch to run.
# It is here to pin down tensor shapes and RoPE semantics before training code exists.

import math

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None
    print("PyTorch is not installed in this environment yet; model code is a design sketch for now.")

if torch is not None:
    def rotate_half(x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def apply_rope(x, move_pos):
        """Apply RoPE with positions shared across slots in the same move.

        x:        [B, H, N, Dh]
        move_pos: [N], where N = context_moves * move_seq_len and positions repeat by slot
        """
        dh = x.shape[-1]
        assert dh % 2 == 0
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dh, 2, device=x.device).float() / dh))
        freqs = torch.einsum("n,d->nd", move_pos.float(), inv_freq)
        emb = torch.repeat_interleave(freqs, repeats=2, dim=-1)  # [N, Dh]
        cos = emb.cos()[None, None, :, :]
        sin = emb.sin()[None, None, :, :]
        return (x * cos) + (rotate_half(x) * sin)

    class RopeSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, dropout=0.0):
            super().__init__()
            assert d_model % n_heads == 0
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            assert self.head_dim % 2 == 0
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out = nn.Linear(d_model, d_model, bias=False)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x, move_pos, attn_mask):
            B, N, D = x.shape
            qkv = self.qkv(x).view(B, N, 3, self.n_heads, self.head_dim)
            q, k, v = qkv.unbind(dim=2)
            q = q.transpose(1, 2)  # [B, H, N, Dh]
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)

            q = apply_rope(q, move_pos)
            k = apply_rope(k, move_pos)

            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores = scores.masked_fill(~attn_mask[None, None, :, :], float("-inf"))
            attn = self.dropout(scores.softmax(dim=-1))
            y = attn @ v
            y = y.transpose(1, 2).contiguous().view(B, N, D)
            return self.out(y)

    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.0):
            super().__init__()
            self.ln1 = nn.LayerNorm(d_model)
            self.attn = RopeSelfAttention(d_model, n_heads, dropout)
            self.ln2 = nn.LayerNorm(d_model)
            self.mlp = nn.Sequential(
                nn.Linear(d_model, mlp_ratio * d_model),
                nn.GELU(),
                nn.Linear(mlp_ratio * d_model, d_model),
                nn.Dropout(dropout),
            )

        def forward(self, x, move_pos, attn_mask):
            x = x + self.attn(self.ln1(x), move_pos, attn_mask)
            x = x + self.mlp(self.ln2(x))
            return x

    def build_move_causal_mask(context_moves, move_seq_len, device):
        """Allow tokens to attend to all slots in current/prior moves, never future moves."""
        move_idx = torch.arange(context_moves, device=device).repeat_interleave(move_seq_len)
        return move_idx[:, None] >= move_idx[None, :]

    class PacketTransformerAR(nn.Module):
        """Pure move-history autoregressive baseline.

        Consumes previous move packets and predicts the next move packet in one shot.
        """
        def __init__(self, vocab_size, move_seq_len=8, d_model=256, n_heads=8, n_layers=6, dropout=0.0):
            super().__init__()
            self.move_seq_len = move_seq_len
            self.token_emb = nn.Embedding(vocab_size, d_model)
            self.slot_emb = nn.Embedding(move_seq_len, d_model)
            self.blocks = nn.ModuleList([
                TransformerBlock(d_model, n_heads, dropout=dropout)
                for _ in range(n_layers)
            ])
            self.ln_f = nn.LayerNorm(d_model)
            self.next_slot_queries = nn.Parameter(torch.randn(move_seq_len, d_model) / math.sqrt(d_model))
            self.head = nn.Linear(d_model, vocab_size, bias=False)

        def forward(self, x_ids):
            """
            x_ids: [B, T, S]
            returns logits: [B, S, V] for next move packet
            """
            B, T, S = x_ids.shape
            assert S == self.move_seq_len
            device = x_ids.device

            slot_ids = torch.arange(S, device=device)
            x = self.token_emb(x_ids) + self.slot_emb(slot_ids)[None, None, :, :]
            x = x.view(B, T * S, -1)

            move_pos = torch.arange(T, device=device).repeat_interleave(S)
            attn_mask = build_move_causal_mask(T, S, device)

            for block in self.blocks:
                x = block(x, move_pos, attn_mask)
            x = self.ln_f(x)

            # Summarize the final context move, then use slot queries to emit next packet slots.
            final_move = x[:, -S:, :].mean(dim=1)                 # [B, D]
            slot_states = final_move[:, None, :] + self.next_slot_queries[None, :, :]  # [B, S, D]
            logits = self.head(slot_states)                       # [B, S, V]
            return logits

    # Shape smoke test
    B, T, S = 2, 16, MOVE_SEQ_LEN
    model = PacketTransformerAR(vocab_size=len(VOCAB), move_seq_len=S, d_model=128, n_heads=4, n_layers=2)
    x_ids = torch.randint(0, len(VOCAB), (B, T, S))
    logits = model(x_ids)
    print("x_ids:", tuple(x_ids.shape))
    print("logits:", tuple(logits.shape), "# [batch, move_seq_len, vocab_size]")


## Verifier network: game-prefix result classifier

First target network from the Scryer paper-understanding ticket.

Goal:

```text
move prefix -> verifier -> logits over [white win, black win, draw]
```

Input is still pure move-history packets for now:

```text
x_ids: [batch, context_moves, 8]
y:     [batch]  # 0 white win, 1 black win, 2 draw
```

No thinker yet. No explicit 64-square board embedding yet.

This is a value/verifier baseline: from an incomplete game prefix, predict the eventual result.

RoPE rule remains per-move:

```text
position(move_i, slot_j) = i
```


In [ ]:
# Verifier model. Requires torch to run.
# This intentionally duplicates a few small utilities from the AR sketch so the cell is self-contained.

import math
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from chessgm.tokenizer import ChessTokenizer, VOCAB

tok = ChessTokenizer()
PAD_ID = tok.token_to_id["<PAD>"]
MOVE_SEQ_LEN = 8
RESULT_TO_LABEL = {"1-0": 0, "0-1": 1, "1/2-1/2": 2}
LABEL_TO_RESULT = {0: "1-0", 1: "0-1", 2: "1/2-1/2"}

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None
    print("PyTorch is not installed in this environment yet; verifier code is ready for a torch environment.")

if torch is not None:
    def rotate_half(x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def apply_rope(x, move_pos):
        # x: [B, H, N, Dh], move_pos: [N]
        dh = x.shape[-1]
        assert dh % 2 == 0
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dh, 2, device=x.device).float() / dh))
        freqs = torch.einsum("n,d->nd", move_pos.float(), inv_freq)
        emb = torch.repeat_interleave(freqs, repeats=2, dim=-1)
        cos = emb.cos()[None, None, :, :]
        sin = emb.sin()[None, None, :, :]
        return (x * cos) + (rotate_half(x) * sin)

    class RopeSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, dropout=0.0):
            super().__init__()
            assert d_model % n_heads == 0
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            assert self.head_dim % 2 == 0
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out = nn.Linear(d_model, d_model, bias=False)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x, move_pos, attn_mask=None):
            B, N, D = x.shape
            qkv = self.qkv(x).view(B, N, 3, self.n_heads, self.head_dim)
            q, k, v = qkv.unbind(dim=2)
            q = q.transpose(1, 2)
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            q = apply_rope(q, move_pos)
            k = apply_rope(k, move_pos)
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            if attn_mask is not None:
                scores = scores.masked_fill(~attn_mask[None, None, :, :], float("-inf"))
            attn = self.dropout(scores.softmax(dim=-1))
            y = attn @ v
            y = y.transpose(1, 2).contiguous().view(B, N, D)
            return self.out(y)

    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.0):
            super().__init__()
            self.ln1 = nn.LayerNorm(d_model)
            self.attn = RopeSelfAttention(d_model, n_heads, dropout)
            self.ln2 = nn.LayerNorm(d_model)
            self.mlp = nn.Sequential(
                nn.Linear(d_model, mlp_ratio * d_model),
                nn.GELU(),
                nn.Linear(mlp_ratio * d_model, d_model),
                nn.Dropout(dropout),
            )

        def forward(self, x, move_pos, attn_mask=None):
            x = x + self.attn(self.ln1(x), move_pos, attn_mask)
            x = x + self.mlp(self.ln2(x))
            return x

    def build_move_causal_mask(context_moves, move_seq_len, device):
        move_idx = torch.arange(context_moves, device=device).repeat_interleave(move_seq_len)
        return move_idx[:, None] >= move_idx[None, :]

    class VerifierTransformer(nn.Module):
        """Prefix result classifier.

        x_ids: [B, T, S]
        logits: [B, 3]
        """
        def __init__(self, vocab_size, move_seq_len=8, d_model=256, n_heads=8, n_layers=6, dropout=0.0):
            super().__init__()
            self.move_seq_len = move_seq_len
            self.token_emb = nn.Embedding(vocab_size, d_model)
            self.slot_emb = nn.Embedding(move_seq_len, d_model)
            self.blocks = nn.ModuleList([
                TransformerBlock(d_model, n_heads, dropout=dropout)
                for _ in range(n_layers)
            ])
            self.ln_f = nn.LayerNorm(d_model)
            self.classifier = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.GELU(),
                nn.Linear(d_model, 3),
            )

        def forward(self, x_ids):
            B, T, S = x_ids.shape
            assert S == self.move_seq_len
            device = x_ids.device
            slot_ids = torch.arange(S, device=device)

            x = self.token_emb(x_ids) + self.slot_emb(slot_ids)[None, None, :, :]
            x = x.view(B, T * S, -1)

            move_pos = torch.arange(T, device=device).repeat_interleave(S)
            attn_mask = build_move_causal_mask(T, S, device)

            for block in self.blocks:
                x = block(x, move_pos, attn_mask)
            x = self.ln_f(x)

            # Use the final move packet as the prefix summary.
            final_move_state = x[:, -S:, :].mean(dim=1)
            return self.classifier(final_move_state)

    # Shape smoke test
    B, T, S = 2, 32, MOVE_SEQ_LEN
    verifier = VerifierTransformer(vocab_size=len(VOCAB), move_seq_len=S, d_model=128, n_heads=4, n_layers=2)
    x_ids = torch.randint(0, len(VOCAB), (B, T, S))
    logits = verifier(x_ids)
    print("x_ids:", tuple(x_ids.shape))
    print("logits:", tuple(logits.shape), "# [batch, 3]")


## Verifier data: sample incomplete prefixes

Training mechanism:

1. Read a completed PGN game.
2. Convert its movetext into fixed-width move packets.
3. Sample a prefix length `t`.
4. Left-pad or crop to `context_moves`.
5. Label with final game result.

This lets the verifier learn:

```text
incomplete game prefix -> eventual result distribution
```


In [ ]:
# Dataset/training sketch for the verifier. Requires torch to actually train.

import random
import re
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RESULT_RE = re.compile(r"\s(1-0|0-1|1/2-1/2|\*)\s*$")


def first_result_from_pgn_text(game_text: str):
    # Prefer Result header if present.
    for line in game_text.splitlines():
        m = re.match(r'^\[Result "(.*)"\]$', line)
        if m:
            return m.group(1)
    # Fallback to movetext ending.
    m = RESULT_RE.search(game_text.strip())
    return m.group(1) if m else None


def iter_pgn_games(path: Path):
    buf = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("[Event ") and buf:
                yield "".join(buf)
                buf = []
            buf.append(line)
        if buf:
            yield "".join(buf)


def pgn_game_to_move_packets(game_text: str, seq_len: int = MOVE_SEQ_LEN):
    movetext = "\n".join(line for line in game_text.splitlines() if not line.startswith("["))
    toks = tok.tokenize_movetext(movetext, include_turn_tokens=True)
    packets = []
    current = []
    for t in toks:
        if t.startswith("RESULT_"):
            break
        current.append(t)
        if t == "<EOM>":
            ids = tok.encode_tokens(current)
            if len(ids) <= seq_len:
                packets.append(ids + [PAD_ID] * (seq_len - len(ids)))
            current = []
    if not packets:
        return None
    return np.asarray(packets, dtype=np.uint16)


def make_verifier_examples_from_pgn(path: Path, context_moves=128, max_games=1000, prefixes_per_game=1, seed=0):
    rng = random.Random(seed)
    xs = []
    ys = []
    for game_i, game_text in enumerate(iter_pgn_games(path)):
        if max_games and game_i >= max_games:
            break
        result = first_result_from_pgn_text(game_text)
        if result not in RESULT_TO_LABEL:
            continue
        packets = pgn_game_to_move_packets(game_text)
        if packets is None or len(packets) < 2:
            continue
        for _ in range(prefixes_per_game):
            t = rng.randint(1, len(packets))
            prefix = packets[:t]
            if len(prefix) >= context_moves:
                x = prefix[-context_moves:]
            else:
                pad_row = np.full((context_moves - len(prefix), MOVE_SEQ_LEN), PAD_ID, dtype=np.uint16)
                x = np.concatenate([pad_row, prefix], axis=0)
            xs.append(x)
            ys.append(RESULT_TO_LABEL[result])
    return np.asarray(xs, dtype=np.uint16), np.asarray(ys, dtype=np.int64)

# Smoke-build examples from the tiny sample file if present.
sample_path = ROOT / "data" / "processed" / "lumbras" / "sample_games.pgn"
if sample_path.exists():
    X_np, y_np = make_verifier_examples_from_pgn(sample_path, context_moves=16, max_games=10, prefixes_per_game=2)
    print("X_np:", X_np.shape, X_np.dtype)
    print("y_np:", y_np.shape, y_np.dtype, y_np.tolist())
else:
    print("sample_games.pgn not found; run scripts/sample_pgn_games.py first if you want a local smoke test.")

if torch is not None and sample_path.exists() and len(X_np):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = VerifierTransformer(vocab_size=len(VOCAB), move_seq_len=MOVE_SEQ_LEN, d_model=128, n_heads=4, n_layers=2).to(device)
    xb = torch.from_numpy(X_np.astype(np.int64)).to(device)
    yb = torch.from_numpy(y_np).to(device)
    logits = model(xb)
    loss = F.cross_entropy(logits, yb)
    print("logits:", tuple(logits.shape))
    print("loss:", float(loss.detach().cpu()))

    # Minimal optimizer step smoke test.
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    print("one optimizer step ok")


## Verifier prefix sampling strategy

Verifier training should not sample prefixes blindly.

A game with `N` moves gives many possible prefixes, but early and late prefixes have very different meaning:

- early prefixes mostly test opening/result priors
- middle prefixes test positional evaluation
- late prefixes test conversion / decisive-state recognition
- very short games are distributionally different from normal late-game positions

So the verifier dataset should support configurable prefix sampling buckets.

Two useful bucket families:

1. Absolute ply buckets

```text
opening:    1-16 plies
early_mid:  17-40
midgame:    41-80
late:       81+
```

2. Fraction-of-game buckets

```text
first_20pct
middle_30pct
late_30pct
final_20pct
```

Training sampler should choose:

```text
bucket -> game with valid prefix in bucket -> prefix length t -> crop/pad to context window
```

This lets us rebalance verifier learning across early/mid/late conditions without materializing every possible prefix.


In [ ]:
# Prefix sampling strategy helpers.
# This cell defines sampling functions only. It does not preprocess the full dataset by itself.

from dataclasses import dataclass
import random
from typing import Literal

BucketMode = Literal["absolute", "fraction"]

@dataclass(frozen=True)
class PrefixBucket:
    name: str
    lo: float  # absolute ply if mode="absolute", fraction if mode="fraction"
    hi: float  # inclusive-ish upper bound; clamped per game
    weight: float = 1.0

ABSOLUTE_PREFIX_BUCKETS = [
    PrefixBucket("opening_1_16", 1, 16, 1.0),
    PrefixBucket("early_mid_17_40", 17, 40, 1.0),
    PrefixBucket("midgame_41_80", 41, 80, 1.0),
    PrefixBucket("late_81_plus", 81, 10**9, 1.0),
]

FRACTION_PREFIX_BUCKETS = [
    PrefixBucket("first_20pct", 0.00, 0.20, 1.0),
    PrefixBucket("middle_30pct", 0.20, 0.50, 1.0),
    PrefixBucket("late_30pct", 0.50, 0.80, 1.0),
    PrefixBucket("final_20pct", 0.80, 1.00, 1.0),
]


def bucket_to_prefix_range(num_moves: int, bucket: PrefixBucket, mode: BucketMode) -> tuple[int, int] | None:
    """Return inclusive [lo, hi] prefix lengths for a game, or None if invalid."""
    if num_moves <= 0:
        return None

    if mode == "absolute":
        lo = int(bucket.lo)
        hi = int(min(bucket.hi, num_moves))
    elif mode == "fraction":
        lo = max(1, int(num_moves * bucket.lo) + 1)
        hi = max(lo, int(num_moves * bucket.hi))
        hi = min(hi, num_moves)
    else:
        raise ValueError(f"unknown bucket mode: {mode}")

    if lo > num_moves or hi < lo:
        return None
    return lo, hi


def valid_buckets_for_game(num_moves: int, buckets: list[PrefixBucket], mode: BucketMode) -> list[PrefixBucket]:
    return [b for b in buckets if bucket_to_prefix_range(num_moves, b, mode) is not None]


def weighted_choice(items, weights, rng: random.Random):
    total = sum(weights)
    if total <= 0:
        raise ValueError("weights must sum positive")
    r = rng.random() * total
    upto = 0.0
    for item, weight in zip(items, weights):
        upto += weight
        if upto >= r:
            return item
    return items[-1]


def sample_prefix_length(
    num_moves: int,
    rng: random.Random,
    buckets: list[PrefixBucket] = FRACTION_PREFIX_BUCKETS,
    mode: BucketMode = "fraction",
) -> tuple[int, str]:
    """Sample a prefix length and return (t, bucket_name)."""
    valid = valid_buckets_for_game(num_moves, buckets, mode)
    if not valid:
        # Fallback: any legal prefix.
        return rng.randint(1, num_moves), "fallback_any"

    bucket = weighted_choice(valid, [b.weight for b in valid], rng)
    lo, hi = bucket_to_prefix_range(num_moves, bucket, mode)
    return rng.randint(lo, hi), bucket.name


def crop_or_left_pad_prefix(packets: np.ndarray, prefix_len: int, context_moves: int, pad_id: int = PAD_ID) -> np.ndarray:
    """Take packets[:prefix_len], then crop/pad to [context_moves, MOVE_SEQ_LEN]."""
    prefix = packets[:prefix_len]
    if len(prefix) >= context_moves:
        return prefix[-context_moves:]
    pad_rows = np.full((context_moves - len(prefix), MOVE_SEQ_LEN), pad_id, dtype=np.uint16)
    return np.concatenate([pad_rows, prefix], axis=0)


def make_bucketed_verifier_examples_from_pgn(
    path: Path,
    context_moves: int = 128,
    max_games: int = 1000,
    prefixes_per_game: int = 1,
    buckets: list[PrefixBucket] = FRACTION_PREFIX_BUCKETS,
    mode: BucketMode = "fraction",
    seed: int = 0,
):
    """Create verifier examples with bucketed prefix sampling.

    Returns:
      X: [num_examples, context_moves, MOVE_SEQ_LEN]
      y: [num_examples]
      meta: list with game index, prefix length, bucket, result
    """
    rng = random.Random(seed)
    xs, ys, meta = [], [], []

    for game_i, game_text in enumerate(iter_pgn_games(path)):
        if max_games and game_i >= max_games:
            break
        result = first_result_from_pgn_text(game_text)
        if result not in RESULT_TO_LABEL:
            continue
        packets = pgn_game_to_move_packets(game_text)
        if packets is None or len(packets) < 2:
            continue

        for _ in range(prefixes_per_game):
            t, bucket_name = sample_prefix_length(len(packets), rng, buckets=buckets, mode=mode)
            x = crop_or_left_pad_prefix(packets, t, context_moves=context_moves)
            xs.append(x)
            ys.append(RESULT_TO_LABEL[result])
            meta.append({
                "game_i": game_i,
                "num_moves": int(len(packets)),
                "prefix_len": int(t),
                "bucket": bucket_name,
                "result": result,
            })

    return np.asarray(xs, dtype=np.uint16), np.asarray(ys, dtype=np.int64), meta

# Example usage when ready to run:
# sample_path = ROOT / "data" / "processed" / "lumbras" / "sample_games.pgn"
# X_np, y_np, meta = make_bucketed_verifier_examples_from_pgn(
#     sample_path,
#     context_moves=128,
#     max_games=100,
#     prefixes_per_game=4,
#     buckets=FRACTION_PREFIX_BUCKETS,
#     mode="fraction",
# )
# print(X_np.shape, y_np.shape, meta[:3])
